# Simple Binary Search Tree


## Class `Node`

This is a basic node for a binary tree. To keep the code simple, the node does not use private fields and therefore it does not require any `get` or `set` methods.


In [ ]:
class Node:
    """A node of a binary tree.

    Fields are public on purpose (no get/set methods) to keep the code simple.
    """
    
    def __init__(self, value:str):
        self.__value: str = value  # payload stored in this node
        self.left = None    # root of the left subtree (smaller values)
        self.right = None   # root of the right subtree (larger values)

    def __str__(self) -> str:
        return str(self.value)

    def __lt__(self, other):
        """Order nodes by their values.

        Only __lt__ is defined. Python derives `a > b` by calling `b < a`, so
        `>` works too. There is no __eq__, so `==` falls back to identity:
        two different nodes holding the same value are NOT equal. That is why
        the tree detects a duplicate as "neither smaller nor larger".
        """
        return self.value < other.value

### Testing the tree node


In [ ]:
h = Node("Hello")
w = Node("World")
e = Node("Hello")

print(h == w)   # False: different objects (identity comparison, no __eq__)
print(h == e)   # False: same value, but still different objects
print(h > w)    # False: Python evaluates w < h, and "World" < "Hello" is False
print(h < w)    # True: "Hello" < "World"

# The Binary Search Tree class

Again, for simplicity, there are no private members in this class.


In [ ]:
class BinarySearchTree:
    """A binary search tree with no duplicates: for every node, values in its
    left subtree are smaller than the node, and values in its right subtree
    are larger. Inserting a value that is already in the tree does nothing.
    """
    
    def __init__(self):
        self.root = None  # top node of the tree; None while the tree is empty
        self.size = 0     # number of nodes in the tree (distinct values)

    def insert(self, value):
        """Wrap `value` in a Node and add it to the tree, unless the value is
        already present.

        The root is always reassigned from the result of insert_rec. This is
        what makes the very first insert work (the tree is empty, so the new
        node becomes the root) and it leaves the root unchanged afterwards.
        """
        new_node = Node(value)
        self.root = self.insert_rec(self.root, new_node)

    def insert_rec(self, current_node, new_node):
        """Insert `new_node` into the subtree rooted at `current_node`.

        Returns the root of that subtree after the insertion, so the caller
        can re-link it. Every path ends in the single return below.
        """
        if current_node is None:
            # Base case: found an empty spot. The new node becomes this
            # subtree; the caller attaches it as a child (or as the root).
            self.size += 1
            current_node = new_node
        elif new_node < current_node:
            # Go left, and re-attach whatever the recursive call returns.
            current_node.left = self.insert_rec(current_node.left, new_node)
        elif new_node > current_node:
            # Go right, and re-attach whatever the recursive call returns.
            current_node.right = self.insert_rec(current_node.right, new_node)
        # Otherwise the value is neither smaller nor larger, so it is a
        # duplicate: do nothing. new_node is discarded and size is unchanged.
        #
        # Return this subtree's root unchanged (or the new node, in the base
        # case) so the parent's left/right link, or self.root, stays valid.
        return current_node

    def search(self, value):
        pass  # Implementation tbd

### What went wrong in the first version of `insert_rec`

The version written in class looked like this:

```python
def insert_rec(self, current_node, new_node):
    if current_node is None:
        current_node = new_node
        self.size += 1
        return current_node
    else:
        if new_node < current_node:
            current_node = current_node.left
            self.insert_rec(current_node, new_node)
        elif new_node > current_node:
            current_node = current_node.right
            self.insert_rec(current_node, new_node)
```

Inserting `"now"` and then `"is"` printed `None` for the root: the tree was empty again. There were three mistakes:

1. **The recursive result was discarded.** `insert_rec` returns the root of the subtree it worked on, but the call `self.insert_rec(current_node, new_node)` ignored it. The new node was created and then lost, because nothing set `current_node.left` or `current_node.right` to it. The fix is `current_node.left = self.insert_rec(current_node.left, new_node)`.
2. **The `else` branch returned nothing.** In Python a function that falls off the end returns `None`. `insert` does `self.root = self.insert_rec(...)`, so the second insert overwrote the root with `None`.
3. **`current_node` was overwritten with its child.** Even with a `return`, the function would have returned the child, not the node it was called on, and the parent would have been dropped from the tree. The fix is to leave `current_node` alone and return it.



In [65]:
bst = BinarySearchTree()

In [ ]:
# Build a tree from a line of text. Duplicate words ("the", "of", "our") are
# not inserted, so size should equal the number of distinct words.
data = "now is the winter of our discontent made glorious summer by this sun of york and all the clouds that loured upon our house in the deep bosom of the ocean buried"
words = data.split()
print(f"Number of words to insert: {len(words)}")
print(f"Number of distinct words: {len(set(words))}")
for word in words:
    bst.insert(word)

print(f"Size of BST: {bst.size}")  # matches the number of distinct words